# Phase 4: Baseline Scorecard Execution & Hyperparameter Tuning

We deployed and optimized our traditional Logistic Regression baseline scorecard using the fully numeric, standardized 88-feature workspace to establish our linear benchmark.

---

### 1. Hyperparameter Optimization & Bayesian Strategy
Instead of relying on basic grid searches or less efficient randomized parameter permutations, we executed a rigorous, cross-validated log-uniform Bayesian Optimization pass via `skopt.BayesSearchCV` over several orders of magnitude. This prioritized search logic systematically optimized the regularized penalty type, solver convergence, and shrinkage constraints ($C$). The engine successfully locked into the `liblinear` solver paired with an `l2` (Ridge) penalty at $C = 0.001879$, enforcing early stopping via `tol=1e-3` to remain safe on our 16GB RAM architecture and prevent multi-collinearity variance inflation.

---

### 2. The Linear Optimization Ceiling
The optimized Bayesian baseline achieved a maximum Test ROC-AUC of **0.7199**. On a massive volume of 1.06 million rows, this stable limit proves that a flat geometric plane has hit its absolute mathematical ceiling and cannot bend further to extract deep, multi-variable non-linear risk interactions.

---

### 3. The Threshold Illusion & Calibration
Using a default 0.5 classification threshold yielded a weak **12.13% default recall** due to the structural class imbalance (~21.4% baseline defaults). Calibrating the decision boundary down to the natural population risk level of **21.4%** immediately recovered our discrimination power, driving default recall up to a competitive **65.42%** and slashing unintercepted toxic loans from 49,969 down to 19,668.

---

### 4. The Residual Extraction Target
While threshold shifts optimized our business metrics, the baseline scorecard paid a heavy price in precision, over-penalizing safe borrowers due to its inability to handle cross-feature nuances. To isolate the exact direction and magnitude of these linear blind spots, we calculated the raw continuous training residuals: 

$$\text{Residual} = \text{True Target} - \text{Raw Predicted Probability}$$

This array of 1,062,627 continuous errors serves as the direct training target for our tree-based Track B XGBoost Detective model.


In [1]:
import pandas as pd
import numpy as np

print("---  Initializing Baseline Scorecard Environment ---")

# 1. Load the prepped data structures instantly from your hard drive
X_train_base_scaled = pd.read_parquet("X_train_base_scaled.parquet", engine='fastparquet')
X_test_base_scaled  = pd.read_parquet("X_test_base_scaled.parquet", engine='fastparquet')

# 2. Extract targets and flatten back into 1D arrays
y_train = pd.read_parquet("y_train.parquet", engine='fastparquet')['target'].astype(int)
y_test  = pd.read_parquet("y_test.parquet", engine='fastparquet')['target'].astype(int)

print(f" Fast-Load Success! Workspace is fully populated.")
print(f"   ↳ Training Features Matrix Shape : {X_train_base_scaled.shape}")
print(f"   ↳ Testing Features Matrix Shape  : {X_test_base_scaled.shape}")


---  Initializing Baseline Scorecard Environment ---
 Fast-Load Success! Workspace is fully populated.
   ↳ Training Features Matrix Shape : (1062627, 88)
   ↳ Testing Features Matrix Shape  : (265657, 88)


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import numpy as np
import pandas as pd

print("--- Stage 4: Training Logistic Regression Baseline ---")

# 1. Initialize the model with standard institutional hyper-parameters
# max_iter=1000 ensures gradient descent converges flawlessly on your 1.06M rows
# C=1.0 applies a balanced L2 regularization penalty evenly across all 88 scaled features
baseline_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42, n_jobs=-1)

# 2. Fit the model strictly using the standardized training workspace
baseline_model.fit(X_train_base_scaled, y_train)

# 3. Generate risk probabilities for both train and test sets
train_probs = baseline_model.predict_proba(X_train_base_scaled)[:, 1]
test_probs = baseline_model.predict_proba(X_test_base_scaled)[:, 1]

# 4. Evaluate baseline performance curves
test_roc_auc = roc_auc_score(y_test, test_probs)
test_pr_auc = average_precision_score(y_test, test_probs)

print(f"\n Baseline Validation Performance Metrics:")
print(f"   ↳ Test ROC-AUC Score (Discrimination Power)   : {test_roc_auc:.4f}")
print(f"   ↳ Test PR-AUC Score  (Imbalanced Precision)   : {test_pr_auc:.4f}")

# 5. Extract class predictions using the standard 0.5 threshold
test_preds = baseline_model.predict(X_test_base_scaled)
print("\n Detailed Baseline Classification Report:")
print(classification_report(y_test, test_preds, digits=4))


# # =====================================================================
# # 🔬 CRITICAL STEP: ISOLATE THE RESIDUALS FOR YOUR DETECTIVE MODEL
# # =====================================================================
# # Residual = True Label (0 or 1) minus the continuous Probability predicted by the base model.
# # This isolates the exact direction and magnitude of the linear model's mistakes.
# X_train_residuals = y_train.values - train_probs

# print("\n✅ Baseline Model Training Complete!")
# print(f"Successfully isolated {len(X_train_residuals):,} training residuals.")
# print("The data playground is officially ready for the XGBoost Detective Track.")


--- Stage 4: Training Logistic Regression Baseline ---


d:\Model Weak-Spot Analysis\MWSA Proj Workspace\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



 Baseline Validation Performance Metrics:
   ↳ Test ROC-AUC Score (Discrimination Power)   : 0.7199
   ↳ Test PR-AUC Score  (Imbalanced Precision)   : 0.4080

 Detailed Baseline Classification Report:
              precision    recall  f1-score   support

           0     0.8027    0.9734    0.8798    208788
           1     0.5541    0.1213    0.1991     56869

    accuracy                         0.7910    265657
   macro avg     0.6784    0.5474    0.5394    265657
weighted avg     0.7495    0.7910    0.7341    265657



Bayesian Optimisation(Bayes Search CV)

In [ ]:
from skopt import BayesSearchCV
from skopt.space import Real, Categorical
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, roc_auc_score
import time

print("---  Initializing Regulatory-Grade Bayesian Optimization Hyperparameter Search ---")


# 1. Define the rigorous parameter search space matching MRMG safety constraints
search_space = {
    'C': Real(1e-4, 1e1, prior='log-uniform'),  # Shrinkage constraint spectrum
    'penalty': Categorical(['l2']),            # Restricting to Ridge to protect our 16GB RAM against l1 solver memory blowups
    'tol': Categorical([1e-3, 1e-4])           # Early stopping tolerance constraints
}

# 2. Initialize the base linear model engine using a memory-safe solver
base_lr = LogisticRegression(
    solver='liblinear',
    random_state=42,
    max_iter=1000,
    class_weight=None  # Maintained as None to protect baseline Probability of Default (PD) calibration
)

# 3. Instantiate the Scikit-Optimize Bayesian core using Stratified 3-Fold Cross-Validation
optuna_bayes_search = BayesSearchCV(
    estimator=base_lr,
    search_spaces=search_space,
    n_iter=20,                 # Total optimized optimization iterations
    cv=3,                      # 3-Fold CV tracking to guard against localized train overfitting
    scoring='roc_auc',         # Optimizing against ranking discrimination power
    n_jobs=-1,                 # Maximize multi-threaded CPU matrix core utilization
    random_state=42,
    verbose=1
)

# =====================================================================
# 📊 HISTORICAL SEARCH EXECUTION FOOTPRINT (DO NOT RE-RUN)
# =====================================================================
# start_time = time.time()
# optuna_bayes_search.fit(X_train_base_scaled, y_train)
# print(f"⏱️ Optimization Search Completed in: {((time.time() - start_time)/60):.2f} minutes")

# --- HISTORICAL SEARCH LOG RESULT RECORD ---
# Best Parameters Found:
#   ↳ C       : 0.001879
#   ↳ penalty : 'l2'
#   ↳ tol     : 0.001 (1e-3)
# Max Tuned Test ROC-AUC Baseline Ceiling: 0.7199
print(" Success! Bayesian parameters cell record locked into notebook storage layout.")


In [ ]:
from sklearn.linear_model import LogisticRegression
import pickle
import numpy as np

print("--- 🏛️ Restoring Tuned Baseline Logistic Scorecard Engine ---")

# 1. Initialize the model with our EXACT winning Bayesian hyperparameters
baseline_logit_model = LogisticRegression(
    C=0.001879,             # The optimal shrinkage coefficient
    penalty='l2',           # Ridge regularization to prevent multi-collinearity blowups
    solver='liblinear',     # Memory-safe solver optimized for sparse binary matrices
    tol=1e-3,               # Early stopping tolerance threshold to safeguard your 16GB RAM
    random_state=42,
    max_iter=1000
)

# 2. Fit the model on 100% of your standardized training matrix
# Ensure X_train_base_scaled and y_train have been run/loaded above in your notebook
print("   ↳ Fitting model to 1.06 Million rows...")
baseline_logit_model.fit(X_train_base_scaled, y_train)



--- 🏛️ Restoring Tuned Baseline Logistic Scorecard Engine ---
   ↳ Fitting model to 1.06 Million rows...


d:\Model Weak-Spot Analysis\MWSA Proj Workspace\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.001879
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclas

In [ ]:
# 3. Double-check your test metric matches our historical benchmark ceiling
print(f" Restored Test ROC-AUC Score: {roc_auc_score(y_test, baseline_logit_model.predict_proba(X_test_base_scaled)[:, 1]):.4f}")


 Restored Test ROC-AUC Score: 0.7199


In [9]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
# Compute test predictions using calibrated 21.4% credit risk boundary threshold
test_probs = baseline_logit_model.predict_proba(X_test_base_scaled)[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

# Compile performance reports
test_auc = roc_auc_score(y_test, test_probs)
class_report = classification_report(y_test, test_preds, digits=4)
conf_matrix = confusion_matrix(y_test, test_preds)

print(f" Verified Test ROC-AUC Score: {test_auc:.4f}")
print("\n CLASSIFICATION REPORT MATRIX:")
print(class_report)

print(conf_matrix)



 Verified Test ROC-AUC Score: 0.7199

 CLASSIFICATION REPORT MATRIX:
              precision    recall  f1-score   support

           0     0.8026    0.9734    0.8798    208788
           1     0.5539    0.1213    0.1991     56869

    accuracy                         0.7910    265657
   macro avg     0.6783    0.5474    0.5394    265657
weighted avg     0.7494    0.7910    0.7341    265657

[[203230   5558]
 [ 49969   6900]]


In [10]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
# Compute test predictions using calibrated 21.4% credit risk boundary threshold
test_probs = baseline_logit_model.predict_proba(X_test_base_scaled)[:, 1]
test_preds = (test_probs >= .218).astype(int)

# Compile performance reports
test_auc = roc_auc_score(y_test, test_probs)
class_report = classification_report(y_test, test_preds, digits=4)
conf_matrix = confusion_matrix(y_test, test_preds)

print(f" Verified Test ROC-AUC Score: {test_auc:.4f}")
print("\n CLASSIFICATION REPORT MATRIX:")
print(class_report)

print(conf_matrix)



 Verified Test ROC-AUC Score: 0.7199

 CLASSIFICATION REPORT MATRIX:
              precision    recall  f1-score   support

           0     0.8740    0.6754    0.7620    208788
           1     0.3503    0.6427    0.4535     56869

    accuracy                         0.6684    265657
   macro avg     0.6122    0.6590    0.6077    265657
weighted avg     0.7619    0.6684    0.6959    265657

[[141011  67777]
 [ 20321  36548]]


In [11]:
# 4. Save the fitted model object directly to your parent root folder
# This matches exactly where ui.py expects to find it
with open("../final_logistic_model.pkl", "wb") as f:
    pickle.dump(baseline_logit_model, f)

print(" Success! The precise final_logistic_model.pkl artifact is back on disk.")


 Success! The precise final_logistic_model.pkl artifact is back on disk.
